# Does seed expansion find the members it was not given?


**The test (instruction 64.12).** Half of a niche's hyperLOPIT markers, whole orthogroups at a time, are given as seeds; the other half are hidden. Seed expansion (random walk with restart over the measured networks and the measurement space) ranks the genome with the hyperLOPIT compartment excluded, and the hidden members are counted at fixed depths. The candidate universe is every gene except the seeds. **What this is not:** the genes that are not markers are not verified negatives, so precision here is a lower bound on how often a ranked gene is truly in the niche.

Built 2026-11-07.

In [1]:
import json, os, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
import pandas as pd
NICHES = ['rhoptries', 'micronemes', 'dense granules', 'IMC', 'apicoplast', 'mitochondrion']
STRATEGIES = ['seed_expansion', 'positive_unlabeled', 'set_enrichment', 'geneset_hunt']
SCRIPT = os.path.join(REPO, 'scripts', 'build_seed_expansion_benchmark_notebook.py')
def run(job):
    strategy, niche = job
    tag = niche.replace(' ', '_')
    out = os.path.join(OUT, f'row_{tag}.json' if strategy == 'seed_expansion' else f'row_{strategy}_{tag}.json')
    if not os.path.exists(out):      # a finished niche is reused, never silently recomputed
        subprocess.run([sys.executable, SCRIPT, '--niche', niche, '--out', out, '--strategy', strategy],
                       check=True, env={**os.environ, 'OMP_NUM_THREADS': '2'})
    row = json.load(open(out)); row.setdefault('strategy', strategy)
    return row
jobs = [(s, n) for s in STRATEGIES for n in NICHES]
with ThreadPoolExecutor(max_workers=len(NICHES)) as pool:
    rows = list(pool.map(run, jobs))
table = pd.DataFrame(rows)
table.to_csv(os.path.join(OUT, 'recovery.tsv'), sep='\t', index=False)
table.pivot(index='niche', columns='strategy', values='recall_at_100')

strategy,geneset_hunt,positive_unlabeled,seed_expansion,set_enrichment
niche,,,,
IMC,0.105,0.474,0.526,0.526
apicoplast,0,0.458,0.292,0.333
dense granules,0.143,0.762,0.571,0.524
micronemes,0,0.786,0.857,0.643
mitochondrion,0.086,0.371,0.343,0.229
rhoptries,0.077,0.769,0.808,0.692


### Five splits, full ranking: positive-unlabelled learning

The fast strategy repeated over five orthogroup splits (seeds 0-4), ranking every gene, with average precision over the whole ranking (hidden members it never reaches count as misses). Mean and range over splits; prevalence is the average precision of a random ranking.

In [2]:
REP = os.path.join(OUT, 'repeats'); os.makedirs(REP, exist_ok=True)
def rep(job):
    seed, niche = job
    out = os.path.join(REP, f"pu_{seed}_{niche.replace(' ', '_')}.json")
    if not os.path.exists(out):
        subprocess.run([sys.executable, SCRIPT, '--niche', niche, '--out', out, '--strategy', 'positive_unlabeled',
                        '--split-seed', str(seed), '--top', '8140'], check=True, env={**os.environ, 'OMP_NUM_THREADS': '2'})
    return json.load(open(out))
with ThreadPoolExecutor(max_workers=len(NICHES)) as pool:
    reps = pd.DataFrame(list(pool.map(rep, [(s, n) for s in range(5) for n in NICHES])))
summary = reps.groupby('niche').agg(splits=('split_seed', 'size'), ap_mean=('average_precision', 'mean'),
    ap_min=('average_precision', 'min'), ap_max=('average_precision', 'max'),
    recall100_mean=('recall_at_100', 'mean'), prevalence=('prevalence', 'mean')).round(3)
summary['ap_lift'] = (summary.ap_mean / summary.prevalence).round(1)
summary.to_csv(os.path.join(OUT, 'pu_repeats_summary.tsv'), sep='\t')
summary

,splits,ap_mean,ap_min,ap_max,recall100_mean,prevalence,ap_lift
niche,,,,,,,
IMC,5,0.198,0.11,0.295,0.579,0.002,99
apicoplast,5,0.159,0.118,0.212,0.466,0.003,53
dense granules,5,0.211,0.169,0.246,0.631,0.003,70.3
micronemes,5,0.192,0.104,0.252,0.607,0.002,96
mitochondrion,5,0.078,0.043,0.116,0.287,0.004,19.5
rhoptries,5,0.485,0.412,0.553,0.776,0.003,161.7
